# Covenant Review Report

Upload a facility agreement PDF and run the five-stage extraction pipeline to produce a single, self-contained `*_review_report.html`.

**Setup** (one-time per Colab runtime)
1. Install the package
2. Configure your Gemini API key
3. Upload the PDF
4. Warm up the Docling runtime

**Pipeline** (run top-to-bottom; re-run any single cell to retry a stage)
- **Step 1 — Ingest**: parse the PDF into structured pages and headings (Docling)
- **Step 2 — Vision TOC**: extract the table of contents with a vision model (Gemini)
- **Step 3 — Scout**: LLM identifies which pages contain each covenant — pre-fetched search results, typically zero tool calls
- **Step 4 — Slicer**: pull just those pages (pure Python, no LLM)
- **Step 5 — Structural Extraction**: deterministic walk of Docling text items — verbatim by construction, no LLM

**Output**: generate, preview inline, and download the report.

---
**Running another facility letter in the same runtime**: re-run **Setup · 3** (Upload PDF), then run Steps 1–5 and Generate again. The upload cell clears prior pipeline state so you cannot accidentally mix documents.

If the runtime restarts or disconnects, re-run setup from the top.

## Setup · 1. Install the package

Run once per fresh Colab runtime. Re-run only after a runtime restart or to pull the latest code from GitHub.

In [ ]:
!pip install -q git+https://github.com/gabrielchan1010/aplma.git

## Setup · 2. Gemini API key

Get a free key at [aistudio.google.com](https://aistudio.google.com/app/apikey).

Store it in Colab Secrets (🔑 icon in the left sidebar):
1. Click **+ Add new secret**
2. **Name**: `GEMINI_API_KEY`
3. **Value**: paste your key
4. Toggle **enable notebook access**
5. Click **Save**

The key is stored securely and never written to disk by this notebook.

In [ ]:
from google.colab import userdata
import importlib, os

secret_name = 'GEMINI_API_KEY'

try:
    secret_value = userdata.get(secret_name)
except Exception as exc:
    raise RuntimeError(
        f"Could not read Colab secret '{secret_name}'. "
        "Open the Secrets panel, ensure the name matches exactly, "
        "and enable notebook access."
    ) from exc

if not secret_value:
    raise RuntimeError(
        f"Colab secret '{secret_name}' is empty. "
        "Check the secret name, value, and notebook-access toggle."
    )

os.environ[secret_name] = secret_value

import aplma.config
importlib.reload(aplma.config)

if not aplma.config.GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is still blank after config reload. "
        "Restart runtime and re-run setup cells."
    )

print(f'✓ Gemini API key loaded from Colab Secrets ({len(aplma.config.GEMINI_API_KEY)} chars)')

## Setup · 3. Upload the PDF

Upload one facility agreement PDF. Re-run this cell to switch to a different document — it clears all prior pipeline state.

In [ ]:
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded.')

SOURCE_FILENAME = next(iter(uploaded))
safe_stem = re.sub(r'[^A-Za-z0-9._-]+', '_', Path(SOURCE_FILENAME).stem).strip('._-') or 'agreement'
PDF_PATH = f'/content/{safe_stem}.pdf'
REPORT_PATH = f'/content/{safe_stem}_review_report.html'

# Clear prior pipeline state so we can't mix documents
for _name in ('doc_state', 'toc', 'specs', 'scout_results', 'slice_results', 'sessions'):
    globals().pop(_name, None)

Path(PDF_PATH).write_bytes(uploaded[SOURCE_FILENAME])
print(f'✓ Saved {SOURCE_FILENAME} → {PDF_PATH} ({len(uploaded[SOURCE_FILENAME]):,} bytes)')
print(f'  Report will be written to {REPORT_PATH}')
print('  Pipeline state cleared — ready to run Step 1.')

## Setup · 4. Warm up Docling runtime

Downloads model/runtime assets the first time a Colab runtime sees this environment. Running this cell before Step 1 makes that setup explicit and separates download time from ingest time.

In [ ]:
from time import perf_counter

print('Setup 4/4 · Initializing Docling runtime...', flush=True)
started = perf_counter()

from docling.document_converter import DocumentConverter
_docling_converter = DocumentConverter()

elapsed = perf_counter() - started
print(f'✓ Docling runtime initialized in {elapsed:.1f}s', flush=True)

## Step 1 — Ingest the PDF

[Docling](https://github.com/docling-project/docling) parses the PDF into a structured document: every page, every heading, every paragraph. The result is cached in memory so later stages don't re-parse.

Typical runtime: **1–3 minutes**. If Colab disconnects mid-ingest, Docling cannot resume — reconnect and re-run this cell.

In [ ]:
import threading
from pathlib import Path
from time import perf_counter
from IPython.display import display
from aplma.report.review_session import ingest_pdf

print(f'Step 1/5 · Ingesting {PDF_PATH} with Docling...', flush=True)
print('  Parsing PDF pages, extracting text and headings. This usually takes 1–3 minutes.', flush=True)

_stop = threading.Event()
def _heartbeat():
    elapsed = 0
    while not _stop.wait(timeout=10):
        elapsed += 10
        print(f'  Still ingesting... ({elapsed}s elapsed)', flush=True)
_hb = threading.Thread(target=_heartbeat, daemon=True)
_hb.start()

started = perf_counter()
try:
    doc_state = ingest_pdf(PDF_PATH)
finally:
    _stop.set()
    _hb.join(timeout=2)

elapsed = perf_counter() - started
print(f'✓ Ingest complete in {elapsed:.1f}s. Document id: {doc_state.document_id}', flush=True)
display(doc_state.html())

## Step 2 — Vision TOC extraction

Sends the table-of-contents pages to a Gemini vision model and returns a clean clause list with physical page numbers. Result is cached — re-running this cell after a successful run is free.

In [ ]:
from time import perf_counter
from aplma.report.review_session import run_vision_toc

print('Step 2/5 · Vision TOC: sending TOC page image(s) to Gemini...', flush=True)
started = perf_counter()
try:
    toc = run_vision_toc(doc_state)
except Exception as exc:
    elapsed = perf_counter() - started
    print(f'✗ Vision TOC failed after {elapsed:.1f}s: {type(exc).__name__}: {exc}', flush=True)
    raise
elapsed = perf_counter() - started
print(f'✓ Vision TOC complete in {elapsed:.1f}s. Extracted {len(toc.entries)} TOC entries.', flush=True)
display(toc.html())

## Step 3 — Scout: find the right pages

Python pre-fetches heading and full-text search results from the spec's `search_terms` and injects them directly into the LLM prompt. The LLM returns tight page ranges immediately — typically with zero tool calls. One additional tool call is available if the pre-fetch doesn't cover what it needs.

In [ ]:
from time import perf_counter
from aplma.report.review_session import run_scout
from aplma.specs.loader import DEFAULT_SPEC_PATH, load_spec

spec_paths = {
    'financial_indebtedness': DEFAULT_SPEC_PATH,
}
specs = {name: load_spec(path) for name, path in spec_paths.items()}

# Preserve successful Scout results across cell reruns.
scout_results = globals().get('scout_results') or {}
for spec_name, spec in specs.items():
    prior = scout_results.get(spec_name)
    if prior is not None and prior.error is None:
        print(f'Step 3/5 · Scout for {spec_name}: cached from previous run, skipping.', flush=True)
        display(prior.html())
        continue
    if prior is not None:
        print(f'Step 3/5 · Scout for {spec_name}: previous attempt failed, retrying...', flush=True)
    else:
        print(f'Step 3/5 · Scout for {spec_name}: asking Gemini to identify relevant page ranges...', flush=True)
    started = perf_counter()
    scout = run_scout(doc_state, toc, spec, spec_name=spec_name)
    elapsed = perf_counter() - started
    scout_results[spec_name] = scout
    if scout.error is not None:
        print(f'✗ Scout failed for {spec_name} after {elapsed:.1f}s: {type(scout.error).__name__}: {scout.error}', flush=True)
    else:
        print(f'✓ Scout complete for {spec_name} in {elapsed:.1f}s: {len(scout.ranges)} range(s), {len(scout.trace)} tool call(s).', flush=True)
    display(scout.html())

## Step 4 — Slicer: pull just those pages

Pure Python — no LLM. Concatenates the Scout-selected pages and builds a **char → page map** so every extracted value can cite the exact physical PDF page.

In [ ]:
from time import perf_counter
from aplma.report.review_session import run_slicer

slice_results = {}
for spec_name in specs:
    print(f'Step 4/5 · Slicer for {spec_name}: fetching Scout-selected page text...', flush=True)
    started = perf_counter()
    slice_ = run_slicer(doc_state, scout_results[spec_name])
    elapsed = perf_counter() - started
    slice_results[spec_name] = slice_
    if slice_.error is not None:
        print(f'✗ Slicer failed for {spec_name} after {elapsed:.1f}s: {type(slice_.error).__name__}: {slice_.error}', flush=True)
    else:
        print(f'✓ Slicer complete for {spec_name} in {elapsed:.1f}s: {len(slice_.slice_content):,} chars from {len(slice_.ranges)} range(s).', flush=True)
    display(slice_.html())

## Step 5 — Structural Extraction

Walks Docling's text items deterministically over the sliced pages. **No LLM call.** Every extraction is verbatim by construction — the parser copies Docling output directly. Each value carries a page reference and char interval for highlight-level grounding.

Typical runtime: **< 1 second** per covenant.

In [ ]:
from time import perf_counter
from aplma.report.review_session import run_structural_extraction

# Preserve successful sessions across cell reruns.
prior_by_name = {s.spec_name: s for s in globals().get('sessions') or []}
sessions = []
for spec_name, spec in specs.items():
    prior = prior_by_name.get(spec_name)
    if prior is not None and prior.extract_error is None and prior.rule is not None:
        print(f'Step 5/5 · Structural extraction for {spec_name}: cached from previous run, skipping.', flush=True)
        sessions.append(prior)
        display(prior.html())
        continue
    if prior is not None:
        print(f'Step 5/5 · Structural extraction for {spec_name}: previous attempt failed, retrying...', flush=True)
    else:
        print(f'Step 5/5 · Structural extraction for {spec_name}: walking Docling text items deterministically...', flush=True)
    started = perf_counter()
    session = run_structural_extraction(
        doc_state, toc,
        scout_results[spec_name],
        slice_results[spec_name],
        spec, spec_name=spec_name,
    )
    elapsed = perf_counter() - started
    sessions.append(session)
    if session.extract_error is not None:
        print(f'✗ Structural extraction failed for {spec_name} after {elapsed:.1f}s: {type(session.extract_error).__name__}: {session.extract_error}', flush=True)
    elif session.rule is None:
        print(f'✗ Structural extraction returned no rule for {spec_name} after {elapsed:.1f}s.', flush=True)
    else:
        carve_outs = getattr(session.rule, 'permitted_carve_outs', []) or []
        print(f'✓ Structural extraction complete for {spec_name} in {elapsed:.1f}s: {len(carve_outs)} carve-out(s) extracted (verbatim).', flush=True)
    display(session.html())

## Generate the final report

Renders everything into a single self-contained HTML — lawyer-relevant extractions and source evidence at the top of each covenant tab; developer detail (Scout trace, Slicer, parser diagnostics) collapsed beneath.

In [ ]:
from pathlib import Path
from time import perf_counter
from aplma.report.html_report import render_report

print('Generating HTML report (including source page snapshots)...', flush=True)
started = perf_counter()
Path(REPORT_PATH).write_text(render_report(sessions), encoding='utf-8')
elapsed = perf_counter() - started
size = Path(REPORT_PATH).stat().st_size
print(f'✓ {REPORT_PATH} written in {elapsed:.1f}s ({size:,} bytes)', flush=True)

## Preview the report inline

Switch tabs at the top to move between covenants. Expand any **Pipeline detail** section to see Scout's trace, the Slicer's char-to-page map, or structural parser diagnostics.

In [ ]:
from IPython.display import IFrame
IFrame(REPORT_PATH, width='100%', height=900)

## Download the report

The HTML is fully self-contained — no external CSS, JS, or images — so it opens anywhere.

In [ ]:
from google.colab import files
files.download(REPORT_PATH)